# RQ2 Strategy Topic Modeling with BERTopic

Goal: model how participants report their attack approach using BERTopic on the `strategy` column from `combined_best_attempt.csv`.

Workflow:
1. Load and clean strategy texts.
2. Fit BERTopic with default settings.
3. Apply outlier reduction.
4. Inspect topic outputs and representative documents.
5. Add human-readable labels and optionally collapse overlapping categories.

In [28]:
from pathlib import Path
import pandas as pd

# Optional display settings for easier inspection in notebooks
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_rows", 200)

## Load `combined_best_attempt` and prepare strategy texts

This cell reads the dataset, validates the required `strategy` column, and creates the document list used by BERTopic.

In [29]:
DATA_PATH = Path("../data/combined_best_attempt.csv")
OUTPUT_DIR = Path("../output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
required_col = "strategies"
if required_col not in df.columns:
    raise ValueError(f"Missing required column: {required_col}")

work = df.copy()
work["strategies_clean"] = work["strategies"].fillna("").astype(str).str.strip()
work = work[work["strategies_clean"].ne("")].copy()
docs = work["strategies_clean"].tolist()

print(f"Rows in file: {len(df):,}")
print(f"Rows with non-empty strategies: {len(work):,}")
work[["strategies_clean"]].head()

Rows in file: 501
Rows with non-empty strategies: 501


,strategies_clean
0,Just tried my best to change the results by making it believeable
1,"I changed the writing style of the person asking, making it less eloquent, and altering the time period that they had been suffering with anxiety and depression. I also had them introduce a very e..."
2,"the most important thing i did was to keep the original story the same and its meaning, but what i changed was to re write it using synonyms and paraphrased it, i adjusted the level of details to ..."
3,"I tried changing the tone of the message slightly. I tried rewriting the message entirely. I tried making it feel more formal, less formal, tried more detail, a little less detail. I got it to dro..."
4,"I tried to change the vague time descriptions to more definite times. I also took out the ""I think"" statements and tried to make the opinions more factual."


## Step 2: Fit BERTopic (default) + reduce outliers

Fit BERTopic with default settings, reduce outliers, and save base topic assignments.

In [30]:
from bertopic import BERTopic

if "docs" not in globals() or "work" not in globals():
    raise RuntimeError("Run Cell 4 first to create 'docs' and 'work'.")

topic_model = BERTopic()
topics, probs = topic_model.fit_transform(docs)

# Outlier reduction (requested setup)
reduced_topics = topic_model.reduce_outliers(docs, topics)
topic_model.update_topics(docs, topics=reduced_topics)
work["topic"] = reduced_topics

topic_info = topic_model.get_topic_info()
topic_info.to_csv(OUTPUT_DIR / "strategy_topic_info.csv", index=False)
work.to_csv(OUTPUT_DIR / "strategy_docs_with_topics.csv", index=False)

print("Saved:")
print(f"- {OUTPUT_DIR / 'strategy_topic_info.csv'}")
print(f"- {OUTPUT_DIR / 'strategy_docs_with_topics.csv'}")

display(topic_info.head(30))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7824.34it/s]
2026-05-19 16:13:39,468 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


Saved:
- ../output/strategy_topic_info.csv
- ../output/strategy_docs_with_topics.csv


,Topic,Count,Name,Representation,Representative_Docs
0,0,201,0_and_more_the_to,"[and, more, the, to, of, statement, make, original, also, as]","[I rephrased the original statement to make it sound more sincere and heartfelt, using phrases like ""unforgettable moment"" and ""incredible"" to convey a stronger sense of emotional authenticity. I ..."
1,1,169,1_the_to_and_more,"[the, to, and, more, of, statement, ai, as, it, make]",[I used a strategy of embellishing and restructuring the original statement to make it sound more elaborate and less spontaneous. The modification involved adding more descriptive language and emo...
2,2,82,2_tried_the_to_it,"[tried, the, to, it, but, and, that, was, my, not]","[I tried using the different words with the same meaning, I tried changing some of the words and phrases. But that did not work, so to use filler words, I tried to keep the core statements yet how..."
3,3,49,3_the_story_to_it,"[the, story, to, it, in, tried, but, details, what, and]","[the most important thing i did was to keep the original story the same and its meaning, but what i changed was to re write it using synonyms and paraphrased it, i adjusted the level of details to..."


## Step 3: Inspect top c-TF-IDF words per topic

Use this as the first interpretation aid before looking at representative documents.

In [31]:
if "topic_model" not in globals() or "topic_info" not in globals():
    raise RuntimeError("Run Step 2 first to fit BERTopic.")

top_words = []
for topic_id in topic_info[topic_info["Topic"] != -1]["Topic"].tolist():
    topic_terms = topic_model.get_topic(topic_id) or []
    for rank, (word, score) in enumerate(topic_terms[:30], start=1):
        top_words.append(
            {"topic": topic_id, "rank": rank, "word": word, "score": score}
        )

top_words_df = pd.DataFrame(top_words)
display(top_words_df.head(100))

top_words_df.to_csv(OUTPUT_DIR / "strategy_topic_top_words.csv", index=False)
print(f"Saved: {OUTPUT_DIR / 'strategy_topic_top_words.csv'}")

,topic,rank,word,score
0,0,1,and,0.125832
1,0,2,more,0.091159
2,0,3,the,0.090847
3,0,4,to,0.090342
4,0,5,of,0.063910
5,0,6,statement,0.059293
6,0,7,make,0.052547
7,0,8,original,0.045012
8,0,9,also,0.043365
9,0,10,as,0.042438


Saved: ../output/strategy_topic_top_words.csv


## Step 4: Inspect representative documents per topic

Use these documents to verify the meaning of each topic before assigning labels.

In [32]:
if "topic_model" not in globals() or "work" not in globals():
    raise RuntimeError("Run Step 2 first to fit BERTopic.")

valid_topics = sorted([t for t in work["topic"].unique() if t != -1])
topic_samples = []

for t in valid_topics:
    rep_docs_raw = topic_model.get_representative_docs(t)
    if isinstance(rep_docs_raw, dict):
        docs_for_t = rep_docs_raw.get(t, [])
    elif isinstance(rep_docs_raw, list):
        docs_for_t = rep_docs_raw
    else:
        docs_for_t = []

    for rank, text in enumerate(docs_for_t[:30], start=1):
        topic_samples.append(
            {"topic": t, "rank": rank, "representative_doc": str(text)}
        )

topic_samples_df = pd.DataFrame(topic_samples)
display(topic_samples_df.head(50))

topic_samples_df.to_csv(OUTPUT_DIR / "strategy_topic_representative_docs.csv", index=False)
print(f"Saved: {OUTPUT_DIR / 'strategy_topic_representative_docs.csv'}")

,topic,rank,representative_doc
0,0,1,"I rephrased the original statement to make it sound more sincere and heartfelt, using phrases like ""unforgettable moment"" and ""incredible"" to convey a stronger sense of emotional authenticity. I a..."
1,0,2,"I used a rephrasing strategy to modify the original statement, replacing informal language and colloquial expressions with more formal and polished vocabulary. This involved changing the tone and ..."
2,0,3,"I rephrased the original statement to make it sound more sincere and authentic, using words and phrases that conveyed a sense of vulnerability and emotional honesty. I also added more detail and c..."
3,1,1,I used a strategy of embellishing and restructuring the original statement to make it sound more elaborate and less spontaneous. The modification involved adding more descriptive language and emot...
4,1,2,I used a strategy of embellishing the emotional tone and language of the original statement to make it appear more sensational and potentially deceptive. This involved adding words and phrases wit...
5,1,3,"I used a strategy of rephrasing the original statement to make it sound more formal and less personal, while preserving the original meaning and content. This involved replacing colloquial express..."
6,2,1,I tried using the different words with the same meaning
7,2,2,"I tried changing some of the words and phrases. But that did not work, so to use filler words"
8,2,3,I tried to keep the core statements yet however tried to change certain words and use synonyms instead to try and get the core text still the same. I changed certain words and use synonyms in thei...
9,3,1,"the most important thing i did was to keep the original story the same and its meaning, but what i changed was to re write it using synonyms and paraphrased it, i adjusted the level of details to ..."


Saved: ../output/strategy_topic_representative_docs.csv


## Step 4b: Sample additional documents per topic

Use these top-ranked documents to see more variation within each topic before labeling.

In [33]:
if "work" not in globals() or "topic_model" not in globals():
    raise RuntimeError("Run Step 2 first to fit BERTopic.")

topic_doc_samples = []
top_n_docs_per_topic = 10

# Reconstruct document-level topic probabilities if available.
doc_info = topic_model.get_document_info(docs)

for topic_id in sorted([t for t in work["topic"].unique() if t != -1]):
    topic_docs = doc_info[doc_info["Topic"] == topic_id].copy()

    if "Probability" in topic_docs.columns:
        topic_docs = topic_docs.sort_values("Probability", ascending=False)

    for rank, (_, row) in enumerate(topic_docs.head(top_n_docs_per_topic).iterrows(), start=1):
        topic_doc_samples.append(
            {
                "topic": topic_id,
                "rank": rank,
                "probability": row.get("Probability", None),
                "document": row.get("Document", ""),
            }
        )

topic_doc_samples_df = pd.DataFrame(topic_doc_samples)
display(topic_doc_samples_df.head(100))

topic_doc_samples_df.to_csv(OUTPUT_DIR / "strategy_topic_additional_docs.csv", index=False)
print(f"Saved: {OUTPUT_DIR / 'strategy_topic_additional_docs.csv'}")

,topic,rank,probability,document
0,0,1,1.0,I tried to make the statement less performative and to make it feel a little more factual. Like i was just stating what happened rather than trying to use emotion to win someone over. I think tryi...
1,0,2,1.0,"I rephrased the original statement to reduce the emphasis on negative emotions and focus on the positive aspects of the experience, using words like ""reconnect"", ""carefree"", and ""uplifting"" to con..."
2,0,3,1.0,"I used a strategy of rephrasing the original statement to make it sound more sincere and heartfelt, while maintaining the same meaning and content. This involved replacing colloquial expressions a..."
3,0,4,1.0,"I used a strategy of embellishing and elaborating on the original statement to make it sound more exaggerated and less genuine. This involved adding more dramatic language and phrases, such as ""un..."
4,0,5,1.0,"I used a strategy of rephrasing the original statement to make it sound more sincere and authentic, by using more formal and polished language. I also added more descriptive phrases to help convey..."
5,0,6,1.0,"I rephrased the original statement to make it sound more sincere and heartfelt, using words like ""lovely"", ""cozy"", and ""grateful"" to convey a sense of warmth and authenticity. I also reorganized t..."
6,0,7,1.0,"I used a strategy of embellishing and dramatizing the original statement to make it appear more sensational and less believable. This involved adding more descriptive language and phrases, such as..."
7,0,8,1.0,I used a strategy of toning down the emotional language and exaggerations in the original statement to make it appear more sincere and truthful. The modification also involved rephrasing the sente...
8,0,9,1.0,"I used a strategy of rephrasing the original statement to make it sound more sincere and heartfelt, while preserving the original meaning and events. This involved replacing straightforward and fa..."
9,0,10,1.0,"I rephrased the original statement to make it sound more sincere and authentic, using phrases like ""life improved significantly"" and ""breathe a sigh of relief"" to convey a sense of genuine emotion..."


Saved: ../output/strategy_topic_additional_docs.csv


## Step 4c: Repeat the topic review by attack modality

Run the same BERTopic, top-word, representative-doc, and additional-doc review separately for `llm` and `human`.
Each subset writes its own output files.

In [ ]:
from bertopic import BERTopic

if "work" not in globals():
    raise RuntimeError("Run Cells 2-4b first.")

if "attack_modality" not in work.columns:
    raise ValueError("Column 'attack_modality' is missing from the working dataframe.")

def run_modality_topic_review(modality, top_n_additional=10):
    subset_work = work[work["attack_modality"].astype(str).str.lower() == modality].copy()
    if subset_work.empty:
        raise ValueError(f"No rows found for attack_modality={modality!r}")

    subset_docs = subset_work["strategies_clean"].tolist()

    subset_topic_model = BERTopic()
    subset_topics, subset_probs = subset_topic_model.fit_transform(subset_docs)

    if -1 in subset_topics:
        subset_reduced_topics = subset_topic_model.reduce_outliers(subset_docs, subset_topics)
        subset_topic_model.update_topics(subset_docs, topics=subset_reduced_topics)
    else:
        subset_reduced_topics = subset_topics
    subset_work["topic"] = subset_reduced_topics

    subset_topic_info = subset_topic_model.get_topic_info()
    subset_top_words = []
    for topic_id in subset_topic_info[subset_topic_info["Topic"] != -1]["Topic"].tolist():
        topic_terms = subset_topic_model.get_topic(topic_id) or []
        for rank, (word, score) in enumerate(topic_terms[:30], start=1):
            subset_top_words.append(
                {"topic": topic_id, "rank": rank, "word": word, "score": score}
            )
    subset_top_words_df = pd.DataFrame(subset_top_words)

    subset_rep_docs = []
    valid_topics = sorted([t for t in subset_work["topic"].unique() if t != -1])
    for topic_id in valid_topics:
        rep_docs_raw = subset_topic_model.get_representative_docs(topic_id)
        if isinstance(rep_docs_raw, dict):
            docs_for_topic = rep_docs_raw.get(topic_id, [])
        elif isinstance(rep_docs_raw, list):
            docs_for_topic = rep_docs_raw
        else:
            docs_for_topic = []

        for rank, text in enumerate(docs_for_topic[:30], start=1):
            subset_rep_docs.append(
                {"topic": topic_id, "rank": rank, "representative_doc": str(text)}
            )
    subset_rep_docs_df = pd.DataFrame(subset_rep_docs)

    subset_doc_info = subset_topic_model.get_document_info(subset_docs)
    subset_additional_docs = []
    for topic_id in valid_topics:
        topic_docs = subset_doc_info[subset_doc_info["Topic"] == topic_id].copy()
        if "Probability" in topic_docs.columns:
            topic_docs = topic_docs.sort_values("Probability", ascending=False)

        for rank, (_, row) in enumerate(topic_docs.head(top_n_additional).iterrows(), start=1):
            subset_additional_docs.append(
                {
                    "topic": topic_id,
                    "rank": rank,
                    "probability": row.get("Probability", None),
                    "document": row.get("Document", ""),
                }
            )
    subset_additional_docs_df = pd.DataFrame(subset_additional_docs)

    subset_topic_info.to_csv(OUTPUT_DIR / f"strategy_topic_info_{modality}.csv", index=False)
    subset_top_words_df.to_csv(OUTPUT_DIR / f"strategy_topic_top_words_{modality}.csv", index=False)
    subset_rep_docs_df.to_csv(OUTPUT_DIR / f"strategy_topic_representative_docs_{modality}.csv", index=False)
    subset_additional_docs_df.to_csv(OUTPUT_DIR / f"strategy_topic_additional_docs_{modality}.csv", index=False)
    subset_work.to_csv(OUTPUT_DIR / f"strategy_docs_with_topics_{modality}.csv", index=False)

    print(f"{modality}: {len(subset_work):,} rows")
    print(f"Saved: {OUTPUT_DIR / ('strategy_topic_info_' + modality + '.csv')}")
    print(f"Saved: {OUTPUT_DIR / ('strategy_topic_top_words_' + modality + '.csv')}")
    print(f"Saved: {OUTPUT_DIR / ('strategy_topic_representative_docs_' + modality + '.csv')}")
    print(f"Saved: {OUTPUT_DIR / ('strategy_topic_additional_docs_' + modality + '.csv')}")
    print(f"Saved: {OUTPUT_DIR / ('strategy_docs_with_topics_' + modality + '.csv')}")

    return {
        "work": subset_work,
        "topic_info": subset_topic_info,
        "top_words": subset_top_words_df,
        "representative_docs": subset_rep_docs_df,
        "additional_docs": subset_additional_docs_df,
    }

modality_results = {}
for modality in ["llm", "human"]:
    modality_results[modality] = run_modality_topic_review(modality=modality, top_n_additional=10)
    display(modality_results[modality]["topic_info"].head(20))
    display(modality_results[modality]["top_words"].head(30))
    display(modality_results[modality]["representative_docs"].head(30))
    display(modality_results[modality]["additional_docs"].head(30))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8907.86it/s]


llm: 278 rows
Saved: ../output/strategy_topic_info_llm.csv
Saved: ../output/strategy_topic_top_words_llm.csv
Saved: ../output/strategy_topic_representative_docs_llm.csv
Saved: ../output/strategy_topic_additional_docs_llm.csv
Saved: ../output/strategy_docs_with_topics_llm.csv


,Topic,Count,Name,Representation,Representative_Docs
0,0,172,0_and_the_to_more,"[and, the, to, more, of, statement, make, original, also, as]","[I used a rephrasing strategy to modify the original statement, replacing informal language and colloquial expressions with more formal and polished vocabulary. This involved changing the tone and..."
1,1,106,1_the_and_to_more,"[the, and, to, more, of, statement, as, make, original, it]",[I used a strategy of embellishing and restructuring the original statement to make it sound more elaborate and less spontaneous. The modification involved adding more descriptive language and emo...


,topic,rank,word,score
0,0,1,and,0.169406
1,0,2,the,0.126328
2,0,3,to,0.121124
3,0,4,more,0.117035
4,0,5,of,0.081514
5,0,6,statement,0.073632
6,0,7,make,0.064895
7,0,8,original,0.055019
8,0,9,also,0.053161
9,0,10,as,0.051228


,topic,rank,representative_doc
0,0,1,"I used a rephrasing strategy to modify the original statement, replacing informal language and colloquial expressions with more formal and polished vocabulary. This involved changing the tone and ..."
1,0,2,"I rephrased the original statement to make it sound more sincere and genuine, using phrases that conveyed a sense of emotional authenticity. I also replaced certain words and phrases with more for..."
2,0,3,"I used a strategy of rephrasing the original statement to make it sound more sincere and authentic, by using more formal and polished language. I also added more descriptive phrases to help convey..."
3,1,1,I used a strategy of embellishing and restructuring the original statement to make it sound more elaborate and less spontaneous. The modification involved adding more descriptive language and emot...
4,1,2,I used a strategy of embellishing the emotional tone and language of the original statement to make it appear more sensational and potentially deceptive. This involved adding words and phrases wit...
5,1,3,"I used a strategy of rephrasing the original statement to make it sound more formal and less personal, while preserving the original meaning and content. This involved replacing colloquial express..."


...

human: 223 rows
Saved: ../output/strategy_topic_info_human.csv
Saved: ../output/strategy_topic_top_words_human.csv
Saved: ../output/strategy_topic_representative_docs_human.csv
Saved: ../output/strategy_topic_additional_docs_human.csv
Saved: ../output/strategy_docs_with_topics_human.csv


,Topic,Count,Name,Representation,Representative_Docs
0,0,172,0_the_to_and_it,"[the, to, and, it, more, tried, that, of, was, in]",[I tried to keep the core statements yet however tried to change certain words and use synonyms instead to try and get the core text still the same. I changed certain words and use synonyms in the...
1,1,51,1_the_to_ai_it,"[the, to, ai, it, and, that, was, so, more, tried]",[I tried to take the orignal context and skew it so it kept the statements' meaning but could be taken as fact instead of deceptive as the AI was predicting. I found out quickly that these adjustm...


,topic,rank,word,score
0,0,1,the,0.138198
1,0,2,to,0.123843
2,0,3,and,0.099419
3,0,4,it,0.077040
4,0,5,more,0.065757
5,0,6,tried,0.063770
6,0,7,that,0.056490
7,0,8,of,0.050471
8,0,9,was,0.048072
9,0,10,in,0.045231


,topic,rank,representative_doc
0,0,1,I tried to keep the core statements yet however tried to change certain words and use synonyms instead to try and get the core text still the same. I changed certain words and use synonyms in thei...
1,0,2,"It was to change the format of the original statement so that the cause of the problem was set out early. Thereafter, try to build a convincing case for why I was feeling as I was and the concerns..."
2,0,3,I tried using more emotive language and making it sound more personal but this did not work. I also tried using more simple language and terms but this also did not work. I really put myself into ...
3,1,1,I tried to take the orignal context and skew it so it kept the statements' meaning but could be taken as fact instead of deceptive as the AI was predicting. I found out quickly that these adjustme...
4,1,2,"I wasn't completely sure how to get an AI to change its prediction. So at first I tried to better summarize the given article, which didn't work. I tried again while removing the Genders from Brot..."
5,1,3,"I added concrete context and emotional contrast to the story to make it more relatable and real.By including details about not being wealthy ,the husband's unusual behavior and the personal meani..."


,topic,rank,probability,document
0,0,1,1.0,I changed siple words to hard words of the same meaning and Rewrote without losing context.
1,0,2,1.0,"I tried to change a few of the key words like truck to car or bike, and wise to a certain group of people such as old, teenagers, and babies. I also tried to add in things that just didn't make se..."
2,0,3,1.0,I tried being more vague and changing specific terms to seem more distant. As well as messing with order and adding slight contradictions that were still true to the main text's body.
3,0,4,1.0,"I tried to make the narration sound distant but still technically accurate, by avoiding too many 'I', and 'my' statements, and then phrased a portion as a question that is still accurate in fact, ..."
4,0,5,1.0,"I chose to eliminate certain details such as, ""she is so beautiful and means the world to me."" I tried to focus more on the details of the accident itself, rather than adding details about the day..."
5,0,6,1.0,I tried including information to make some statements contradictory. I also tried to make some stuff sound over the top.
6,0,7,1.0,I focused on making the language clearer and more straightforward while keeping the key facts the same. I replaced dramatic phrasing with more detailed statements
7,0,8,1.0,I added some uncertainty and then I tried contradicting myself while still mnaking sure that the facts were still intact. I also added a casual lol which made the narration sound less confident w...
8,0,9,1.0,"At first, I tried changing the whole thing to sound more natural, especially given all of the writing errors. This made some progress, but I quickly realized that smaller edits often made larger c..."
9,0,10,1.0,Tried to use more personalised and descriptive writing.


## Step 5: Apply final human-readable strategy codes

Map topic IDs to your final concept labels and numeric codes, then export a coded version of `combined_best_attempt`.

In [ ]:
if "df" not in globals() or "work" not in globals() or "topic" not in work.columns:
    raise RuntimeError("Run Steps 1-4 first to create topic assignments.")

# Your final topic interpretation map
topic_label_map = {
    0: "authenticity",
    1: "emotion",
    2: "rephrasing",
    3: "details",
}

# Numeric strategy code map (can be changed later if needed)
topic_numeric_map = {
    0: 0,
    1: 1,
    2: 2,
    3: 3,
}

coded_df = df.copy()

# Bring topic assignments from the modeled working frame back to the full dataset
coded_df["topic"] = pd.NA
coded_df.loc[work.index, "topic"] = work["topic"].values

coded_df["strategy_coded"] = coded_df["topic"].map(topic_label_map)
coded_df["strategy_coded_num"] = coded_df["topic"].map(topic_numeric_map).astype("Int64")

output_coded_path = OUTPUT_DIR / "combined_best_attempt_strategy_coded.csv"
coded_df.to_csv(output_coded_path, index=False)

print(f"Saved: {output_coded_path}")
coded_df[["strategies", "topic", "strategy_coded", "strategy_coded_num"]].head(40)

Saved: ../output/combined_best_attempt_strategy_coded.csv


,strategies,topic,strategy_coded,strategy_coded_num
0,Just tried my best to change the results by making it believeable,2,rephrasing,2
1,"I changed the writing style of the person asking, making it less eloquent, and altering the time period that they had been suffering with anxiety and depression. I also had them introduce a very e...",1,emotion,1
2,"the most important thing i did was to keep the original story the same and its meaning, but what i changed was to re write it using synonyms and paraphrased it, i adjusted the level of details to ...",3,details,3
3,"I tried changing the tone of the message slightly. I tried rewriting the message entirely. I tried making it feel more formal, less formal, tried more detail, a little less detail. I got it to dro...",2,rephrasing,2
4,"I tried to change the vague time descriptions to more definite times. I also took out the ""I think"" statements and tried to make the opinions more factual.",2,rephrasing,2
5,"I added uncertainty phrases like 'not entirely sure' and 'maybe', questioned my own memory, and offered alternative interpretations. I also tried to make the narrative less consistent to reduce cr...",2,rephrasing,2
6,Began to alter sentences which kind of worked but I rephrased everything in the end and did small alterations to make it believe it,2,rephrasing,2
7,In the most successful attempt I tried to make the language almost more casual I guess? Rearranged the story and language a bit and tried to imagine how I'd phrase it if it was true for me.,3,details,3
8,"Initially to try and reduce some of the details, sometimes over explanation can create an air of deception. Later it was to perhaps be more conversational almost? However I had no clue what I coul...",1,emotion,1
9,"I just wanted to tweak so details as the instructions stated but it seemed with every attempt to make it more deceitful, the AI found it more truthful. I wish the could have been more flexibility ...",1,emotion,1
